# Algorithm Comparison

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

Helpers loaded (retina mode).


## Algorithm Comparison

Run all algorithms on the same input and compare the number of character comparisons.

In [2]:
def compute_sp(P):
    """Compute sp[j] = length of longest proper border of P[0..j]."""
    m = len(P)
    sp = [0] * m
    k = 0
    for j in range(1, m):
        while k > 0 and P[k] != P[j]:
            k = sp[k - 1]
        if P[k] == P[j]:
            k += 1
        sp[j] = k
    return sp


def bm_bad_char_table(P):
    """Last occurrence of each character in P (R function)."""
    R = {}
    for i, c in enumerate(P):
        R[c] = i
    return R


def count_naive(T, P):
    count = 0
    for s in range(len(T) - len(P) + 1):
        for j in range(len(P)):
            count += 1
            if T[s + j] != P[j]:
                break
    return count


def count_kmp(T, P):
    sp = compute_sp(P)
    count = 0
    j = 0
    for i in range(len(T)):
        while j > 0 and T[i] != P[j]:
            count += 1
            j = sp[j - 1]
        count += 1
        if T[i] == P[j]:
            j += 1
            if j == len(P):
                j = sp[j - 1]
    return count


def count_bm(T, P):
    n, m = len(T), len(P)
    R = bm_bad_char_table(P)
    count = 0
    s = 0
    while s <= n - m:
        j = m - 1
        while j >= 0:
            count += 1
            if P[j] != T[s + j]:
                break
            j -= 1
        if j < 0:
            s += 1
        else:
            bad = R.get(T[s + j], -1)
            s += max(1, j - bad)
    return count


def draw_comparison(T, P):
    naive_c = count_naive(T, P)
    kmp_c = count_kmp(T, P)
    bm_c = count_bm(T, P)

    algos = ["Naive", "KMP", "Boyer-Moore"]
    counts = [naive_c, kmp_c, bm_c]
    colors = ["#EF9A9A", "#90CAF9", "#A5D6A7"]

    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.barh(algos, counts, color=colors, edgecolor="#555", height=0.55)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(c), va="center", fontsize=13, fontweight="bold")

    ax.set_xlabel("Character comparisons", fontsize=12)
    ax.set_title(f"T = \"{T}\"   P = \"{P}\"   |T|={len(T)}  |P|={len(P)}", fontsize=12)
    ax.set_xlim(0, max(counts) * 1.2)
    safe_tight_layout(fig)
    plt.show()


T_cmp = widgets.Text(value="abcaabcabcaab", description="T:", layout=widgets.Layout(width="500px"))
P_cmp = widgets.Text(value="abcab", description="P:", layout=widgets.Layout(width="500px"))
out_cmp = widgets.Output()


def refresh_cmp(_=None):
    T, P = T_cmp.value, P_cmp.value
    if not T or not P or len(P) > len(T):
        return
    with out_cmp:
        clear_output(wait=True)
        draw_comparison(T, P)


T_cmp.observe(refresh_cmp, "value")
P_cmp.observe(refresh_cmp, "value")
display(T_cmp, P_cmp, out_cmp)
refresh_cmp()

Text(value='abcaabcabcaab', description='T:', layout=Layout(width='500px'))

Text(value='abcab', description='P:', layout=Layout(width='500px'))

Output()